In [1]:
from rapidfuzz import process, fuzz
import pandas as pd
import numpy as np
from pathlib import Path
import re

In [2]:
path_dflb = Path(r"E:\ProyectoAnalisisElectrico\BarrasEstaciones\LocatedBars.csv") # df_located_bars
path_dfp = Path(r"E:\ProyectoAnalisisElectrico\DiaPromedio\Periodos\2505_2604\2505_2604_mean_period.parquet")
dflb = pd.read_csv(path_dflb, sep=";", encoding="utf-8")
dfp = pd.read_parquet(path_dfp).reset_index()

In [3]:
dflb.head()

,Nombre Subestación,ID,Nombre,Nombre Centro Control,Nombre Propietario,Nombre Coordinado,Número,Nemotecnico,Descripcion,ID_E,Número_E,Nemotecnico_E,Región,Provincia,Comuna,Macrozona
0,S/E CENTRAL ALFALFAL,1,BA S/E CENTRAL ALFALFAL 12KV BP1,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,1,BA01G0010SE001G0010,NaN,199,1,SE001G0010,Metropolitana de Santiago,Cordillera,San José de Maipo,Centro
1,S/E CENTRAL ALFALFAL,2,BA S/E CENTRAL ALFALFAL 12KV BP2,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,2,BA02G0010SE001G0010,NaN,199,1,SE001G0010,Metropolitana de Santiago,Cordillera,San José de Maipo,Centro
2,S/E CENTRAL MAITENES,6,BA S/E CENTRAL MAITENES 6.6KV B1,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,1,BA01G0010SE004G0010,NaN,201,4,SE004G0010,Metropolitana de Santiago,Cordillera,San José de Maipo,Centro
3,S/E CENTRAL QUELTEHUES,7,BA S/E CENTRAL QUELTEHUES 110KV BP1,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,1,BA01G0010SE006G0010,NaN,203,6,SE006G0010,Metropolitana de Santiago,Cordillera,San José de Maipo,Centro
4,S/E CENTRAL QUELTEHUES,8,BA S/E CENTRAL QUELTEHUES 12KV,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,2,BA02G0010SE006G0010,NaN,203,6,SE006G0010,Metropolitana de Santiago,Cordillera,San José de Maipo,Centro


In [4]:
df = dflb[['Nombre',"Nombre Subestación", "Macrozona", "Región"]].copy()

In [5]:
df.head()

,Nombre,Nombre Subestación,Macrozona,Región
0,BA S/E CENTRAL ALFALFAL 12KV BP1,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago
1,BA S/E CENTRAL ALFALFAL 12KV BP2,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago
2,BA S/E CENTRAL MAITENES 6.6KV B1,S/E CENTRAL MAITENES,Centro,Metropolitana de Santiago
3,BA S/E CENTRAL QUELTEHUES 110KV BP1,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago
4,BA S/E CENTRAL QUELTEHUES 12KV,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago


vamos a buscar alguna forma de rescatar las palabras clave

In [6]:
df['NS1'] = df['Nombre Subestación'].str.replace(r'^S/E\s*', '', regex=True)
df['N1'] = df['Nombre'].str.replace(r'BA\s+S/E\s*', '', regex=True)

df['NS1'] = df['NS1'].str.replace(r'\(.*?\)', '', regex=True).str.strip()
df['N1'] = df['N1'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

df['N1'] = df['N1'].str.replace(r'\d+(?:[.,]\d+)?\s*[kK][vV].*', '', regex=True).str.strip()

df['NS1'] = df['NS1'].str.replace(r'(?i)\bC\b\s*', '', regex=True).str.strip()

df['NS1'] = df['NS1'].str.replace(r'(?i)TAP\s+OFF', 'T.OFF', regex=True)
# Eliminamos los números seguidos de KV en la columna NS1
df['NS1'] = df['NS1'].str.replace(r'(?i)\s*\d+\s*KV', '', regex=True).str.strip()
# Reemplazamos cualquier tipo de espacio (uno o varios) por nada (texto vacío)
df['NS1'] = df['NS1'].str.replace(r'\s+', '', regex=True)

In [7]:
df["Voltaje"] = df['Nombre'].str.extract(r'(\d+(?:[.,]\d+)?)\s*[kK][vV]')
df.loc[df['Nombre'].isin(['BA S/E MONTURAQUI 4.16 BP1', 'BA S/E MONTURAQUI 4.16 BP2']), 'Voltaje'] = "4.16"
df.loc[df['Nombre'] == 'BA S/E CODELCO VENTANAS 2 110 BP1', 'Voltaje'] = "110.0"
df['Voltaje'] = df['Voltaje'].str.replace(',', '.')
df['Voltaje'] = df['Voltaje'].astype(float)
df = df.dropna(subset=['Voltaje'])
df["Voltaje"] = np.trunc(df["Voltaje"]).astype(int)

In [8]:
df.head()

,Nombre,Nombre Subestación,Macrozona,Región,NS1,N1,Voltaje
0,BA S/E CENTRAL ALFALFAL 12KV BP1,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago,CENTRALALFALFAL,CENTRAL ALFALFAL,12
1,BA S/E CENTRAL ALFALFAL 12KV BP2,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago,CENTRALALFALFAL,CENTRAL ALFALFAL,12
2,BA S/E CENTRAL MAITENES 6.6KV B1,S/E CENTRAL MAITENES,Centro,Metropolitana de Santiago,CENTRALMAITENES,CENTRAL MAITENES,6
3,BA S/E CENTRAL QUELTEHUES 110KV BP1,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago,CENTRALQUELTEHUES,CENTRAL QUELTEHUES,110
4,BA S/E CENTRAL QUELTEHUES 12KV,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago,CENTRALQUELTEHUES,CENTRAL QUELTEHUES,12


In [9]:
df['NS2'] = df['NS1'].astype(str) + "_" + df['Voltaje'].astype(str)
df.head()

,Nombre,Nombre Subestación,Macrozona,Región,NS1,N1,Voltaje,NS2
0,BA S/E CENTRAL ALFALFAL 12KV BP1,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago,CENTRALALFALFAL,CENTRAL ALFALFAL,12,CENTRALALFALFAL_12
1,BA S/E CENTRAL ALFALFAL 12KV BP2,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago,CENTRALALFALFAL,CENTRAL ALFALFAL,12,CENTRALALFALFAL_12
2,BA S/E CENTRAL MAITENES 6.6KV B1,S/E CENTRAL MAITENES,Centro,Metropolitana de Santiago,CENTRALMAITENES,CENTRAL MAITENES,6,CENTRALMAITENES_6
3,BA S/E CENTRAL QUELTEHUES 110KV BP1,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago,CENTRALQUELTEHUES,CENTRAL QUELTEHUES,110,CENTRALQUELTEHUES_110
4,BA S/E CENTRAL QUELTEHUES 12KV,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago,CENTRALQUELTEHUES,CENTRAL QUELTEHUES,12,CENTRALQUELTEHUES_12


In [10]:
conteo_regiones = df.groupby('NS2')['Región'].nunique()
ns2_multiples = conteo_regiones[conteo_regiones > 1]

# 2. Eliminamos las filas problemáticas usando el símbolo '~' (NO están en ns2_multiples)
df = df[~df['NS2'].isin(ns2_multiples.index)]

# 3. Reseteamos el índice para mantener el orden de la tabla
df = df.reset_index(drop=True)

In [11]:
opciones_validas = df['NS2'].dropna().unique().tolist()

def obtener_top_5(nombre_buscar):
    if pd.isna(nombre_buscar):
        return pd.Series([None, 0] * 5)
        
    resultados = process.extract(
        str(nombre_buscar), 
        opciones_validas, 
        scorer=fuzz.token_sort_ratio,
        limit=5
    )
    
    fila_resultado = []
    for match in resultados:
        fila_resultado.extend([match[0], match[1]])
        
    while len(fila_resultado) < 10:
        fila_resultado.extend([None, 0])
        
    return pd.Series(fila_resultado)

dfu = dfp[['nombre_barra', 'tension']].dropna(subset=['nombre_barra']).drop_duplicates(subset=['nombre_barra']).copy()

dfu = dfu.reset_index(drop=True)

dfu['nombre_compare'] = dfu['nombre_barra'].astype(str) + "_" + dfu['tension'].astype(str)

columnas_top = [
    'Match_1', 'Score_1', 
    'Match_2', 'Score_2', 
    'Match_3', 'Score_3', 
    'Match_4', 'Score_4', 
    'Match_5', 'Score_5'
]

dfu[columnas_top] = dfu['nombre_compare'].apply(obtener_top_5)


In [12]:
pd.reset_option('display.max_rows')
dfu

,nombre_barra,tension,nombre_compare,Match_1,Score_1,Match_2,Score_2,Match_3,Score_3,Match_4,Score_4,Match_5,Score_5
0,DOMEYKO,220,DOMEYKO_220,DOMEYKO_220,100.000000,DONGOYO_220,72.727273,CONEJO_220,66.666667,SVCDOMEYKO_19,66.666667,DONHECTOR_220,66.666667
1,ALTONORTE,110,ALTONORTE_110,ALTONORTE_110,100.000000,ALTONORTE_13,88.000000,ALTOBONITO_110,74.074074,CONDORES_110,72.000000,POZOALMONTE_110,71.428571
2,ANTOFAGASTA,13,ANTOFAGASTA_13,ANTOFAGASTA_13,100.000000,ANTOFAGASTA_110,89.655172,PLANTAMATTA_13,71.428571,PLANTAS_13,66.666667,SANTAMARTA_12,66.666667
3,ARICA,66,ARICA_66,ARICA_66,100.000000,VILLARRICA_66,76.190476,PARINACOTA_66,76.190476,TALCA_66,75.000000,ARICA_13,75.000000
4,CHAPIQUINA,13,CHAPIQUINA_13,QUINTA_13,72.727273,MARIQUINA_23,72.000000,CHARRUA_13,69.565217,CENTRALCHAPIQUIÑA_3,68.750000,CHENA_13,66.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,LA_POLVORA,110,LA_POLVORA_110,LAPOLVORA_110,96.296296,LAPOLVORA_13,84.615385,LAPOLVORA_220,81.481481,LAPORTADA_110,74.074074,ELSALVADOR_110,71.428571
489,SULFUROS,220,SULFUROS_220,SULFUROS_220,100.000000,SULFUROS_13,78.260870,SULFUROS_69,78.260870,FUTURO_220,72.727273,SANLUIS_220,69.565217
490,RECINTO,23,RECINTO_23,RECINTO_23,100.000000,RECINTO_33,90.000000,CENTRO_23,73.684211,TRESPINOS_23,72.727273,RIOBONITO_23,72.727273
491,TMELON,12,TMELON_12,ELMELON_12,84.210526,TUNELELMELON_12,75.000000,MEJILLONES_2,66.666667,ELBATO_12,66.666667,SANTAELENA_12,63.636364


In [13]:
tolerancia = 90

df_dudosos = dfu[dfu['Score_1'] < tolerancia].sort_values('Score_1')

print(f"Tienes {len(df_dudosos)} barras que requieren revisión manual.")
df_dudosos.to_excel('auditoria_cruces_norevisada.xlsx', index=False)

Tienes 192 barras que requieren revisión manual.


In [14]:
auditoria_path = Path(r"E:\ProyectoAnalisisElectrico\BarrasEstaciones\auditoria_cruces.xlsx")

In [15]:
dfa = pd.read_excel(auditoria_path)


In [16]:
dfa.head()

,nombre_barra,tension,nombre_compare,Match_1
0,M.V.CEN.,154,M.V.CEN._154,NaN
1,S.F.MOSTAZAL,13,S.F.MOSTAZAL_13,NaN
2,S.P.PAPELES,66,S.P.PAPELES_66,NaN
3,QTILCOCO,13,QTILCOCO_13,NaN
4,ENLACE_SNG,220,ENLACE_SNG_220,T.OFFENLACE_220


In [17]:
dfx = dfu.merge(
    dfa[['nombre_barra', "tension",  'Match_1']], 
    on='nombre_barra', 
    how='left', 
    suffixes=('', '_auditoria')
)

fue_auditada = dfx['nombre_barra'].isin(dfa['nombre_barra'])

dfx['Match_Definitivo'] = np.where(fue_auditada, dfx['Match_1_auditoria'], dfx['Match_1'])


In [18]:
df.columns, dfx.columns

(Index(['Nombre', 'Nombre Subestación', 'Macrozona', 'Región', 'NS1', 'N1',
        'Voltaje', 'NS2'],
       dtype='str'),
 Index(['nombre_barra', 'tension', 'nombre_compare', 'Match_1', 'Score_1',
        'Match_2', 'Score_2', 'Match_3', 'Score_3', 'Match_4', 'Score_4',
        'Match_5', 'Score_5', 'tension_auditoria', 'Match_1_auditoria',
        'Match_Definitivo'],
       dtype='str'))

In [20]:
dfx_puente = dfx[['Match_Definitivo', 'nombre_barra', "tension"]].dropna(subset=['Match_Definitivo']).drop_duplicates(subset=['Match_Definitivo'])

# 2. Hacemos el cruce hacia df
dfk = df.merge(
    dfx_puente,
    left_on='NS2',
    right_on='Match_Definitivo',
    how='left'
)

# 3. Borramos la columna 'Match_Definitivo' que se acaba de pegar, ya que es idéntica a 'NS2' y solo hace ruido
dfk = dfk.drop(columns=['Match_Definitivo'])

dfk = dfk.dropna(subset=['nombre_barra']).reset_index(drop=True)

# 4. Verificamos cómo quedó tu df con la nueva columna al final
dfk.head()

,Nombre,Nombre Subestación,Macrozona,Región,NS1,N1,Voltaje,NS2,nombre_barra,tension
0,BA S/E CENTRAL QUELTEHUES 12KV,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago,CENTRALQUELTEHUES,CENTRAL QUELTEHUES,12,CENTRALQUELTEHUES_12,QUELTEHUES,12.0
1,BA S/E CENTRAL PILMAIQUEN 13.2KV,S/E CENTRAL PILMAIQUEN,Sur,Los Ríos,CENTRALPILMAIQUEN,CENTRAL PILMAIQUEN,13,CENTRALPILMAIQUEN_13,PILMAIQUEN,13.0
2,BA S/E ENLACE 66KV,S/E ENLACE,Centro Sur,Biobío,ENLACE,ENLACE,66,ENLACE_66,ENLACE,66.0
3,BA S/E DEGAÑ 23KV B1,S/E DEGAÑ,Sur,Los Lagos,DEGAÑ,DEGAÑ,23,DEGAÑ_23,DEGAN,13.0
4,BA S/E CALERA CENTRO 62KV,S/E CALERA CENTRO,Centro,Valparaíso,CALERACENTRO,CALERA CENTRO,62,CALERACENTRO_62,CALERA.C,62.0


In [ ]:
# 1. Creamos el diccionario geográfico desde dfk, asegurando que cada barra aparezca solo una vez
dfk_puente = dfk[['nombre_barra', 'Macrozona', 'Región']].drop_duplicates(subset=['nombre_barra'])

# 2. Limpiamos dfp por si acaso (borramos Macrozona y Región si ya existían de un intento anterior)
for col in ['Macrozona', 'Región']:
    if col in dfp.columns:
        dfp = dfp.drop(columns=[col])

# 3. Hacemos el cruce final hacia tus 3000 filas
dfp = dfp.merge(
    dfk_puente, 
    on='nombre_barra', 
    how='left'
)

# 4. Verificación de éxito
print(f"Tu dfp se mantiene con {len(dfp)} filas.")
display(dfp[['nombre_barra', 'Macrozona', 'Región']].head(10))

# Extra: Ver cuántas quedaron sin información geográfica
faltantes = dfp['Región'].isna().sum()

Tu dfp se mantiene con 164352 filas.


,nombre_barra,Macrozona,Región
0,DOMEYKO,Norte,Antofagasta
1,DOMEYKO,Norte,Antofagasta
2,DOMEYKO,Norte,Antofagasta
3,DOMEYKO,Norte,Antofagasta
4,DOMEYKO,Norte,Antofagasta
5,DOMEYKO,Norte,Antofagasta
6,DOMEYKO,Norte,Antofagasta
7,DOMEYKO,Norte,Antofagasta
8,DOMEYKO,Norte,Antofagasta
9,DOMEYKO,Norte,Antofagasta



Filas en dfp que no encontraron Macrozona/Región: 11904


In [22]:
dfp[dfp['Región'].isna()]

,clave,Zona,Hora,medida,CMg[CLP/KWh],valorizado_CLP,Calendario_Activo,RUT,rut_log,n_ruts,...,n_nombres_cortos,nombre_barra,nombre_barra_log,n_nombres_barra,tension,tension_log,n_tensiones,period,Macrozona,Región
144,00600100128RA,Norte,0,-263.839127,65.521940,-17575.152373,111111111111,88.006.900-4,88.006.900-4,1,...,1,CHAPIQUINA,CHAPIQUINA,1,13,13,1,2505_2604,NaN,NaN
145,00600100128RA,Norte,1,-247.323466,65.338795,-16379.261186,111111111111,88.006.900-4,88.006.900-4,1,...,1,CHAPIQUINA,CHAPIQUINA,1,13,13,1,2505_2604,NaN,NaN
146,00600100128RA,Norte,2,-239.693040,66.725081,-16212.598162,111111111111,88.006.900-4,88.006.900-4,1,...,1,CHAPIQUINA,CHAPIQUINA,1,13,13,1,2505_2604,NaN,NaN
147,00600100128RA,Norte,3,-235.686187,67.530219,-16086.867238,111111111111,88.006.900-4,88.006.900-4,1,...,1,CHAPIQUINA,CHAPIQUINA,1,13,13,1,2505_2604,NaN,NaN
148,00600100128RA,Norte,4,-233.985348,67.487338,-15909.715142,111111111111,88.006.900-4,88.006.900-4,1,...,1,CHAPIQUINA,CHAPIQUINA,1,13,13,1,2505_2604,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163291,UTFSM_JM CARRERA,Norte Distribución,19,-87.953350,60.100509,-6696.031445,111111111111,77.209.283-0,77.209.283-0,1,...,1,MIRAFLORES,MIRAFLORES,1,13,13,1,2505_2604,NaN,NaN
163292,UTFSM_JM CARRERA,Norte Distribución,20,-98.319366,71.767231,-7483.542161,111111111111,77.209.283-0,77.209.283-0,1,...,1,MIRAFLORES,MIRAFLORES,1,13,13,1,2505_2604,NaN,NaN
163293,UTFSM_JM CARRERA,Norte Distribución,21,-97.824405,71.847159,-7240.109052,111111111111,77.209.283-0,77.209.283-0,1,...,1,MIRAFLORES,MIRAFLORES,1,13,13,1,2505_2604,NaN,NaN
163294,UTFSM_JM CARRERA,Norte Distribución,22,-90.553199,71.942490,-6602.507977,111111111111,77.209.283-0,77.209.283-0,1,...,1,MIRAFLORES,MIRAFLORES,1,13,13,1,2505_2604,NaN,NaN


In [23]:
dfp.to_parquet("periodwfc.parquet", engine="pyarrow", compression="snappy")